<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2025-Tutorial-Notebooks/blob/main/tutorials_notebooks_in_class_2025/W08_intro_to_hugging_face_transformers_datasets.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>


# Quick Introduction to Huggingface's Transformers- and Datasets-Library

Adjusted from: https://huggingface.co/transformers/training.html

- Transformers docs: https://huggingface.co/transformers/index.html


In [1]:
!pip install transformers datasets

import os
os.environ["WANDB_MODE"] = "disabled"

#Loading the dataset

In [2]:
import pandas as pd
import datasets
dataset = datasets.load_dataset('sms_spam')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

In [3]:
print(dataset.keys())

dict_keys(['train'])


In [4]:
print(len(dataset['train']))

5574


In [5]:
# next time, if we only want a few examples:
dataset = datasets.load_dataset('sms_spam', split='train[800:1000]')  # [:100] [:1%]

In [6]:
from collections import Counter
Counter(dataset['label'])

Counter({0: 165, 1: 35})

In [7]:
dataset

Dataset({
    features: ['sms', 'label'],
    num_rows: 200
})

In [8]:
dataset[0]

{'sms': '"Gimme a few" was  &lt;#&gt;  minutes ago\n', 'label': 0}

In [9]:
dataset[1]

{'sms': 'Last Chance! Claim ur £150 worth of discount vouchers today! Text SHOP to 85023 now! SavaMob, offers mobile! T Cs SavaMob POBOX84, M263UZ. £3.00 Sub. 16\n',
 'label': 1}

#Loading a HuggingFace Transformer

First part:

## **Tokenizer:**

The tokenizer is responsible for converting human-readable text into the numerical format that the model can understand. Each model has been trained with a specific type of tokenizer, so it’s crucial to use the correct tokenizer to ensure the input text is processed in a way that the model expects. The tokenizer:

* Splits text into tokens: Breaks down text into individual words or subwords,  depending on the model (e.g., BERT uses WordPiece, while GPT-2 uses Byte-Pair Encoding).

* Maps tokens to IDs: Converts each token to an integer ID that represents it in the model’s vocabulary. This ensures that each word or subword has a corresponding, unique numerical representation.

* Handles special tokens: Adds tokens that indicate sentence boundaries, padding, or start-of-sequence markers, which can be essential for tasks like translation, summarization, or question answering.

* If you use a different tokenizer from the one the model was trained on, the token IDs will not match what the model expects, resulting in poor or incorrect predictions.

Each model architecture uses a slightly (sometimes significantly) different tokenizer. Depending on the model we use, we need to load the right tokenizer (else nothing works correctly).

Here is how to load it for bert-base.

In [10]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

There is a convenient abstraction to avoid looking to find the right tokenizer, Bert, Roberta, XLM-RoBERTa : AutoTokenizer

In [11]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

In [12]:
tokenizer(dataset[0]['sms'])

{'input_ids': [101, 107, 144, 4060, 3263, 170, 1374, 107, 1108, 111, 181, 1204, 132, 108, 111, 176, 1204, 132, 1904, 2403, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [13]:
tokenizer(dataset[0]['sms'], return_tensors="pt", padding='max_length', truncation=True, max_length=128)

{'input_ids': tensor([[ 101,  107,  144, 4060, 3263,  170, 1374,  107, 1108,  111,  181, 1204,
          132,  108,  111,  176, 1204,  132, 1904, 2403,  102,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0,

**input_ids**
Contains token IDs representing the input text.
Starts with [CLS] (101) and ends with [SEP] (102).
Each ID corresponds to a specific token or subword in BERT's vocabulary.

**token_type_ids**
Distinguishes segments within the input.
For single-sequence inputs, all values are 0.
For paired inputs, 0 for the first segment, 1 for the second.

**attention_mask**
Indicates tokens to be attended to with 1 and padding with 0.

In [14]:
encoded_dataset = [tokenizer(item['sms'], return_tensors="pt", padding='max_length', truncation=True, max_length=128) for item in dataset]

In [15]:
import torch
for enc_item, item in zip(encoded_dataset, dataset):
    enc_item['labels'] = torch.LongTensor([item['label']])

In [16]:
print(len(encoded_dataset))
for key, val in encoded_dataset[0].items():
    print(f'key: {key}, dimensions: {val.size()}')

200
key: input_ids, dimensions: torch.Size([1, 128])
key: token_type_ids, dimensions: torch.Size([1, 128])
key: attention_mask, dimensions: torch.Size([1, 128])
key: labels, dimensions: torch.Size([1])


In [17]:
from random import shuffle
shuffle(encoded_dataset)

Second part:

## **Model Weights**

Model Architecture: The structure of the neural network (e.g., transformer layers, attention heads) specific to the model type (like BERT or GPT-2).

Pre-trained Weights: Learned parameters from pre-training on large datasets, enabling the model to perform tasks like classification or summarization without starting from scratch.

Model Configuration: Settings such as hidden layer size, number of layers, and dropout rates that control the model’s behavior and performance.

The randomly initialised (or trained) classification/regression head already attached to the end of the model model. The architecture of it is specified by the end part of the model. - ForSequenceClassification

Hint: For the exercise, it's a token classification task so we use -ForTokenClassification

Here is how to load it for bert-base. Depending on the model size, the size can quickly stack up

In [18]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
model = BertForSequenceClassification.from_pretrained('bert-base-cased', num_labels=2)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


There is a convenient abstraction to avoid looking to find the right model too: AutoModel

In [19]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained('bert-base-cased', num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Finetuning Methods

In [20]:
train_set = encoded_dataset[:100]
test_set = encoded_dataset[100:]

### Traditional Torch Finetuning

In [22]:
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=1e-5)
model.train()  # set model train state
outputs = model(**train_set[0])[0]
print(outputs)
loss = outputs
loss.backward()
optimizer.step()

tensor(0.9040, grad_fn=<NllLossBackward0>)


### HuggingFace Trainer

In [23]:
# we don't need the batch dimension when using the trainer
# because the trainer does batching for us
for item in encoded_dataset:
    for key in item:
        item[key] = torch.squeeze(item[key])
train_set = encoded_dataset[:100]
test_set = encoded_dataset[100:]

In [24]:
training_args = TrainingArguments(
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    output_dir='results',
    logging_dir='logs',
    no_cuda=False,  # defaults to false anyway, just to be explicit
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_set,
)

/tmp/ipython-input-2569550662.py:10: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [25]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


TrainOutput(global_step=25, training_loss=0.4020393753051758, metrics={'train_runtime': 168.3112, 'train_samples_per_second': 0.594, 'train_steps_per_second': 0.149, 'total_flos': 6577776384000.0, 'train_loss': 0.4020393753051758, 'epoch': 1.0})

In [26]:
preds = trainer.predict(test_set)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [27]:
print(preds.predictions[:2]) # raw logits for the first two samples
print(preds.predictions[:2].argmax(-1)) # argmax on the logits --> predicted labels
print(preds.label_ids[:2]) # gold labels
print(preds.metrics)

[[ 0.3491196  -1.2866421 ]
 [-0.07378548  0.678182  ]]
[0 1]
[0 1]
{'test_loss': 0.1957024335861206, 'test_runtime': 38.6762, 'test_samples_per_second': 2.586, 'test_steps_per_second': 0.646}


### Evaluation

In [28]:
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
predictions = preds.predictions.argmax(-1)
f1_score(preds.label_ids, predictions, average='binary')

0.9696969696969697

In [29]:
confusion_matrix(predictions, preds.label_ids)

array([[83,  1],
       [ 0, 16]])